In [29]:
"""
SV callset preprocessing notebook
"""

import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2/model"
os.chdir(workdir)
os.getcwd()

import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from pysam import VariantFile

import svensemble.vcfparse as vcfp
import svensemble._caller_params as caller_params
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.utils as sv_utils

importlib.reload(caller_params)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(sv_utils)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

preprocessed_savedir = Path('saved/preprocessed/')
preprocessed_savedir.mkdir(parents=True, exist_ok=True)

callsets = {
    'gnomad': {
        'source': 'gnomad',
        'path': Path('data/gnomad.v4.1.sv.sites.vcf.gz'),
        'params': caller_params.gnomad_params
    },
    'gridss': {
        'source': 'gridss',
        'path': Path('data/test_gridss_output.NA12878.vcf'),
        'params': caller_params.gridss_params
    },
    'manta_candidate': {
        'source': 'manta',
        'path': Path('data/candidateSV.vcf.gz'),
        'params': caller_params.manta_params
    },
    'manta_diploid': {
        'source': 'manta',
        'path': Path('data/diploidSV.vcf.gz'),
        'params': caller_params.manta_params
    }
}

to_process_list = ['gnomad', 'gridss', 'manta_candidate', 'manta_diploid']

In [ ]:
if 'gridss' in to_process_list:
    gridss_vcf = VariantFile(callsets['gridss']['path'])
    gridss_df, gridss_meta = vcfp.parse_vcf_to_df(vcf_file=gridss_vcf, field_info=callsets['gridss']['params'])
    gridss_df = caller_specific.gridss_preprocess(gridss_df, verbose=True)
    gridss_bp_df = bnds.bnd_to_bp(gridss_df)
    gridss_bp_df.to_csv(preprocessed_savedir / 'gridss_bps.csv')
    del gridss_df, gridss_meta, gridss_bp_df


In [ ]:
if 'gnomad' in to_process_list:
    gnomad_vcf = VariantFile(callsets['gnomad']['path'])
    gnomad_df, gnomad_meta = vcfp.parse_vcf_to_df(vcf_file=gnomad_vcf, field_info=callsets['gnomad']['params'])
    gnomad_df = caller_specific.gnomad_preprocess(gnomad_df, verbose=True)
    gnomad_bnd_df = caller_specific.gnomad_event_to_bnd(gnomad_df)
    gnomad_bp_df = bnds.bnd_to_bp(gnomad_bnd_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    gnomad_bp_df.to_csv(preprocessed_savedir / 'gnomad_bps.csv')
    del gnomad_df, gnomad_meta, gnomad_bnd_df, gnomad_bp_df


In [ ]:
if 'manta_diploid' in to_process_list:
    manta_diploid_vcf = VariantFile(callsets['manta_diploid']['path'])
    manta_diploid_df, manta_diploid_meta = vcfp.parse_vcf_to_df(vcf_file=manta_diploid_vcf, field_info=callsets['manta_diploid']['params'])
    manta_bnd_only_df = manta_diploid_df[manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_evt_only_df = manta_diploid_df[~manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_bnd_only_df)
    manta_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_evt_only_df)
    manta_evt_bnd_df = evts.event_to_bnd(manta_evt_only_df)
    manta_diploid_df = pd.concat([manta_bnd_only_df, manta_evt_bnd_df], axis=0)
    manta_diploid_bp_df = bnds.bnd_to_bp(manta_diploid_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    manta_diploid_bp_df.to_csv(preprocessed_savedir / 'manta_diploid_bps.csv')
    del manta_diploid_df, manta_diploid_meta, manta_bnd_only_df, manta_evt_only_df, manta_evt_bnd_df, manta_diploid_bp_df

In [ ]:
if 'manta_candidate' in to_process_list:
    manta_candidate_vcf = VariantFile(callsets['manta_candidate']['path'])
    manta_candidate_df, manta_candidate_meta = vcfp.parse_vcf_to_df(vcf_file=manta_candidate_vcf, field_info=callsets['manta_candidate']['params'])
    manta_candidate_bnd_only_df = manta_candidate_df[manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_candidate_evt_only_df = manta_candidate_df[~manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_candidate_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_candidate_bnd_only_df)
    manta_candidate_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_candidate_evt_only_df)
    manta_candidate_evt_bnd_df = evts.event_to_bnd(manta_candidate_evt_only_df)
    manta_candidate_df = pd.concat([manta_candidate_bnd_only_df, manta_candidate_evt_bnd_df], axis=0)
    manta_candidate_bp_df = bnds.bnd_to_bp(manta_candidate_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    manta_candidate_bp_df.to_csv(preprocessed_savedir / 'manta_candidate_bps.csv')
    del manta_candidate_df, manta_candidate_meta, manta_candidate_bnd_only_df, manta_candidate_evt_only_df, manta_candidate_evt_bnd_df, manta_candidate_bp_df